In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import (
    SystemMessage, RemoveMessage, HumanMessage, BaseMessage
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    summary: str


def chat_node(state: State):
    summary = state.get("summary", "")
    messages = []
    if summary:
        messages.append(SystemMessage(content=f"Conversation summary: {summary}"))
    messages = messages + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


def summarize_node(state: State):
    existing = state.get("summary", "")
    if existing:
        prompt = (f"This is the existing summary: {existing}\n\n"
                  "Extend it using the new conversation above.")
    else:
        prompt = "Summarize the conversation above."

    resp = llm.invoke(state["messages"] + [HumanMessage(content=prompt)])

    # keep only the last 2 messages, delete the rest
    to_delete = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": resp.content, "messages": to_delete}


def should_summarize(state: State):
    return len(state["messages"]) > 6


builder = StateGraph(State)
builder.add_node("chat", chat_node)
builder.add_node("summarize", summarize_node)
builder.add_edge(START, "chat")
builder.add_conditional_edges("chat", should_summarize, {True: "summarize", False: END})
builder.add_edge("summarize", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [2]:
config = {"configurable": {"thread_id": "1"}}


def show_state():
    snap = graph.get_state(config).values
    print("SUMMARY:", snap.get("summary", "") or "(empty)")
    print("---")
    for m in snap["messages"]:
        print("-", type(m).__name__, ":", str(m.content)[:50])
    print("total messages:", len(snap["messages"]))

In [3]:
questions = [
    "What is quantum physics?",
    "How is Einstein related to it?",
    "What are his famous works?",
    "Explain special relativity.",
]

for q in questions:
    graph.invoke({"messages": [HumanMessage(content=q)]}, config=config)

show_state()

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


SUMMARY: [{'type': 'text', 'text': 'Here is a summary of our conversation, broken down by the four main topics we covered:\n\n---\n\n### 1. What is Quantum Physics?\nQuantum physics is the study of matter and energy at the smallest scale (atoms and subatomic particles). Unlike our everyday world, the quantum world is governed by **probabilities rather than certainties**. \n* **Key Principles:** Energy comes in packets (*quanta*); objects behave as both waves and particles; particles can exist in multiple states at once (*superposition*); measuring one particle can instantly affect an *entangled* partner far away; and you cannot simultaneously know a particle\'s exact location and speed (*uncertainty principle*).\n* **Impact:** It powers modern technology like microchips, lasers, MRIs, and solar panels.\n\n### 2. Einstein\'s Relationship with Quantum Physics\nEinstein had a paradoxical relationship with quantum theory—he was both a **reluctant father** and its **biggest critic**.\n* **T